### Imports and Load

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
import os
import joblib


In [20]:
processed_path = Path.cwd().parent / "data" / "processed"

df = pd.read_csv(processed_path / "engineered_dataset.csv")
print(f"Loaded shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head(10)

Loaded shape: (31500, 131)
Label distribution:
label
2    10500
1    10500
0    10500
Name: count, dtype: int64


,scenario_id,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,...,rolling_mean_velocity_b,rolling_std_velocity_b,rolling_mean_pressure_drop_bc,rolling_std_pressure_drop_bc,rolling_mean_midpoint_pressure_deviation,rolling_std_midpoint_pressure_deviation,rolling_mean_midpoint_velocity_deviation,rolling_std_midpoint_velocity_deviation,rolling_mean_midpoint_vof_deviation,rolling_std_midpoint_vof_deviation
0,blockage_25_loc16m,1,35793.499725,2.534141,0.034635,24.489446,11.197253,0.401012,19965.653843,2.474028,...,2.474028,0.000000,15575.231918,0.000000,-126.306983,0.000000,-0.056303,0.000000,-0.001483,0.000000
1,blockage_25_loc16m,2,36620.208522,2.484805,0.038275,25.658411,10.641124,0.405089,20589.416498,2.421521,...,2.447775,0.037128,15784.963909,296.605826,-72.177522,76.550617,-0.066276,0.014104,-0.002528,0.001478
2,blockage_25_loc16m,3,36251.794105,2.473708,0.034279,25.294662,9.484428,0.398871,20946.810911,2.531422,...,2.475657,0.054968,16232.488105,803.007614,255.640536,570.371841,-0.013167,0.092527,0.000671,0.005639
3,blockage_25_loc16m,4,35694.151655,2.478258,0.032199,27.022206,9.489620,0.393614,19637.008191,2.406712,...,2.458421,0.056592,16108.980877,700.640394,151.894868,509.838549,-0.032113,0.084518,0.002760,0.006216
4,blockage_25_loc16m,5,35834.383052,2.467021,0.037261,22.894842,10.660725,0.400522,19776.413014,2.442424,...,2.455221,0.049530,16234.319894,668.372701,189.286487,449.379767,-0.035610,0.073611,0.001922,0.005700
5,blockage_25_loc16m,6,36787.297566,2.569537,0.035647,23.439768,10.250242,0.398309,19106.601335,2.521981,...,2.464812,0.058000,16196.748693,718.113403,-14.784149,755.761004,-0.038142,0.074710,0.001523,0.006056
6,blockage_25_loc16m,7,36144.736489,2.485088,0.034797,24.908137,10.644589,0.395470,20094.551930,2.449309,...,2.470369,0.054009,16101.232826,780.737717,-64.481335,764.152646,-0.026778,0.071726,0.002178,0.005521
7,blockage_25_loc16m,8,35566.008814,2.496601,0.034091,24.837599,10.843902,0.395164,19447.852780,2.483421,...,2.460769,0.043732,15913.531313,551.465133,-239.649376,556.816344,-0.048303,0.031828,-0.000082,0.005327
8,blockage_25_loc16m,9,36058.571722,2.555000,0.033721,24.339944,9.959091,0.393493,20766.372402,2.433175,...,2.466062,0.036569,16149.602407,692.414566,-45.119414,733.695701,-0.042645,0.024456,-0.001548,0.002397
9,blockage_25_loc16m,10,36087.612877,2.511439,0.037675,25.088742,10.973236,0.402177,19771.627334,2.479944,...,2.473566,0.034284,15879.582168,669.599339,-205.931085,716.409973,-0.041884,0.024244,-0.001061,0.002660


### Define Columns

In [21]:
# Metadata columns — not features, not target
metadata_cols = ["scenario_id", "timestep"]

# Dashboard/DB columns — keep in dataset but not ML features
non_ml_cols = ["fault_type", "effect_factor"]

# Target
target_col = "label"

# ML feature columns — everything else
exclude_from_features = metadata_cols + non_ml_cols + [target_col]
feature_cols = [c for c in df.columns if c not in exclude_from_features]

print(f"Total ML features : {len(feature_cols)}")
print(f"Target            : {target_col}")
print(f"Metadata kept     : {metadata_cols}")
print(f"Non-ML kept       : {non_ml_cols}")

Total ML features : 126
Target            : label
Metadata kept     : ['scenario_id', 'timestep']
Non-ML kept       : ['fault_type', 'effect_factor']


### Scenario Level Split


In [22]:
# Get unique scenario per class
scenarios_normal   = df[df["label"] == 0]["scenario_id"].unique()
scenarios_leak     = df[df["label"] == 1]["scenario_id"].unique()
scenarios_blockage = df[df["label"] == 2]["scenario_id"].unique()

print(f"Normal scenarios   : {len(scenarios_normal)}")
print(f"Leak scenarios     : {len(scenarios_leak)}")
print(f"Blockage scenarios : {len(scenarios_blockage)}")

def split_scenarios(scenarios, seed =42):
    train, temp = train_test_split(scenarios, test_size = 0.25, random_state = seed)
    val, test = train_test_split(temp, test_size = 0.50, random_state = seed)

    return train, val, test

train_n, val_n, test_n = split_scenarios(scenarios_normal)
train_l, val_l, test_l = split_scenarios(scenarios_leak)
train_b, val_b, test_b = split_scenarios(scenarios_blockage)

# combine scenario lists
train_scenarios = np.concatenate([train_n, train_l, train_b])
val_scenarios = np.concatenate([val_n, val_l, val_b])
test_scenarios = np.concatenate([test_n, test_l, test_b])

# create splits
df_train = df[df["scenario_id"].isin(train_scenarios)]
df_val = df[df["scenario_id"].isin(val_scenarios)]
df_test = df[df["scenario_id"].isin(test_scenarios)]

print(f"\nTrain shape : {df_train.shape} | labels: {df_train['label'].value_counts().to_dict()}")
print(f"Val shape   : {df_val.shape}   | labels: {df_val['label'].value_counts().to_dict()}")
print(f"Test shape  : {df_test.shape}  | labels: {df_test['label'].value_counts().to_dict()}")


Normal scenarios   : 15
Leak scenarios     : 15
Blockage scenarios : 15

Train shape : (25200, 131) | labels: {2: 8400, 1: 8400, 0: 8400}
Val shape   : (4200, 131)   | labels: {2: 1400, 1: 1400, 0: 1400}
Test shape  : (2100, 131)  | labels: {2: 700, 1: 700, 0: 700}


- We are splittitng by scenario and not random rows because if we are to split randomly timesteps 1-699 of a scenario end up in train and 700 ends up in testing.
- The model basically would have een the scansion during training leading to data leakage.
- Scenario level split guarantees all 700 timesteps of a scenario stay in the same split.

### Separate feature and targets

In [ ]:
X_train = df_train[feature_cols].copy()
y_train = df_train[target_col].copy()




In [ ]:
# Check for data quality before scaling
print("Checking for issues before scaling")

inf_count = np.isinf(X_train.select)